# Handle Data Storage

### Create a list of all Zettel

In [1]:
import os
import regex as re
import pandas as pd
import cProfile
import pstats

from zettelsortierung.DataTypes import *
from zettelsortierung.Transformation import *
from zettelsortierung.RegionDetection import OpenCVRegionDetector
from zettelsortierung.OCR import OCR, OCRpreprocessing, OCRpostprocessing
from zettelsortierung.Visualisation import vis_boxes_labels

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
root = '../data/processed/Scans.parquet'
sammlung = Zettelsammlung.from_parquet(root, k=100)
sammlung = set([zettel.recto for zettel in sammlung])

detector = OpenCVRegionDetector()

box_probe = Probe()
text_probe = Probe()

pipeline = \
Composition(
    Batch(1000),
    SequentialApp(
        ParallelApp(
            PatchDetect(detector)
        ),
        Flatten(),
        ParallelApp(
            BoundingBoxPad(10),
        ),
        Store(box_probe),
        ParallelApp(
            CutOutPatch(),
            ResizePatch(),
            BGR2RGB()
        ),
        Sort(key=lambda item: item.feature.shape[1]),
        Batch(16),
        ParallelApp(OCRpreprocessing()),
        SequentialApp(OCR()),
        ParallelApp(OCRpostprocessing()),
        Flatten(),
        Store(text_probe)
    ),
    Flatten()
)

In [3]:
res = pipeline(sammlung)

In [4]:
# cProfile.run('pipeline(sammlung)', 'profile_stats')
# stats = pstats.Stats("profile_stats")
# stats.sort_stats("time").print_stats()

In [5]:
print(box_probe)
print(len(box_probe))
print(text_probe)
print(len(text_probe))

Probe([DataPoint(scan=Scan(A01-00000041_1), feature_id=0, feature=BoundingBox(x=np.int32(1555), y=np.int32(1062), w=np.int32(328), h=np.int32(82))),
DataPoint(scan=Scan(A01-00000087_1), feature_id=2, feature=BoundingBox(x=np.int32(403), y=np.int32(715), w=np.int32(1495), h=np.int32(178))),
DataPoint(scan=Scan(A01-00000087_1), feature_id=3, feature=BoundingBox(x=0, y=np.int32(756), w=np.int32(340), h=np.int32(100))),
DataPoint(scan=Scan(A01-00000087_1), feature_id=4, feature=BoundingBox(x=np.int32(888), y=np.int32(869), w=np.int32(95), h=np.int32(86))),
DataPoint(scan=Scan(A01-00000087_1), feature_id=5, feature=BoundingBox(x=np.int32(433), y=np.int32(876), w=np.int32(205), h=np.int32(92))),
DataPoint(scan=Scan(A01-00000087_1), feature_id=6, feature=BoundingBox(x=np.int32(721), y=np.int32(971), w=np.int32(820), h=np.int32(129))),
DataPoint(scan=Scan(A01-00000063_1), feature_id=0, feature=BoundingBox(x=np.int32(1127), y=np.int32(1011), w=np.int32(612), h=np.int32(102))),
DataPoint(scan=Sc

In [6]:
# Build lookup dictionary from second list
text_lookup = {(scan, feat_id): text for scan, feat_id, text in text_probe}

# Join
pairs = [
    (scan, feat_id, box, text_lookup[(scan, feat_id)])
    for scan, feat_id, box in box_probe
    if (scan, feat_id) in text_lookup
]

In [7]:
counter = 0
number = 10
zettel_set = set([dp.scan for dp in box_probe])
for scan in zettel_set:
    if counter >= number:
        break
    boxes = []
    labels = []
    for pair in pairs:
        if pair[0] == scan:
            boxes.append(pair[2])
            labels.append(pair[3])
    vis_boxes_labels(scan, boxes, labels)
    counter += 1
